In [ ]:
# COMP541 organized folder path helper
from pathlib import Path
import os

def _comp541_phase_dir():
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == 'code':
        return cwd.parent
    if (cwd / 'Input').exists() and (cwd / 'Output').exists():
        return cwd
    if (cwd.parent / 'Input').exists() and (cwd.parent / 'Output').exists():
        return cwd.parent
    # Fallback for the intended layout: notebook/script is inside a Code folder.
    return cwd.parent if cwd.name.lower() == 'code' else cwd

PHASE_DIR = _comp541_phase_dir()
INPUT_DIR = PHASE_DIR / 'Input'
OUTPUT_DIR = PHASE_DIR / 'Output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('INPUT_DIR :', INPUT_DIR.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())


In [ ]:
import pandas as pd
import time
import os
from fast_flights import FlightData, Passengers, create_filter, get_flights_from_filter

In [2]:
ORIGIN = "LAX"
DEPARTURE_DATES = ["2026-07-15", "2026-10-15"]
OUTPUT_FILES = {
    "2026-07-15": OUTPUT_DIR / "flight_data_peak.csv",
    "2026-10-15": OUTPUT_DIR / "flight_data_off_peak.csv"
}
MAX_RETRIES = 3
BASE_DELAY = 5

target_destinations = [
    "CDG", "LHR", "FCO", "MAD", "BCN", "AMS", "ATH", "DUB", "BER", "LIS", "PRG", "VIE", "ZRH", "CPH",
    "NRT", "HND", "KIX", "ICN", "BKK", "SIN", "DPS", "SGN", "HKG", "TPE", "DXB", "IST",
    "MEX", "CUN", "SJD", "PVR", "SJU", "PUJ", "GUA", "SJO", "BOG", "LIM", "EZE", "GRU", "GIG",
    "LAS", "JFK", "MIA", "MCO", "HNL", "YVR", "YYZ",
    "SYD", "MEL", "AKL", "NAN"
]

total_cities = len(target_destinations)
print(f"Destinations loaded: {total_cities} | Dates: {DEPARTURE_DATES}")

Destinations loaded: 50 | Dates: ['2026-07-15', '2026-10-15']


In [3]:
for departure_date in DEPARTURE_DATES:
    extracted_records = []
    output_file = OUTPUT_FILES[departure_date]

    print(f"\nStarting extraction for {departure_date} -> {output_file}")
    print("-" * 50)

    for index, destination in enumerate(target_destinations):
        success = False

        for attempt in range(MAX_RETRIES):
            print(f"[{index + 1}/{total_cities}] {ORIGIN} -> {destination} (Attempt {attempt+1}/{MAX_RETRIES})...")

            try:
                route_segment = [FlightData(date=departure_date, from_airport=ORIGIN, to_airport=destination)]
                flight_filter = create_filter(
                    flight_data=route_segment,
                    trip="one-way",
                    seat="economy",
                    passengers=Passengers(adults=1)
                )

                result = get_flights_from_filter(flight_filter)

                if result and result.flights:
                    def parse_price(flight_obj):
                        p_val = getattr(flight_obj, 'price', None)
                        if p_val is None: return float('inf')
                        clean_str = str(p_val).replace('$', '').replace(',', '').strip()
                        return float(clean_str) if clean_str.replace('.', '', 1).isdigit() else float('inf')

                    cheapest_flight = min(result.flights, key=parse_price)

                    extracted_records.append({
                        "Origin_Hub": ORIGIN,
                        "Destination_Hub": destination,
                        "Departure_Date": departure_date,
                        "Flight_Price_USD": getattr(cheapest_flight, 'price', 'N/A'),
                        "Carrier": getattr(cheapest_flight, 'name', getattr(cheapest_flight, 'airline', 'Unknown')),
                        "Duration_String": getattr(cheapest_flight, 'duration', 'N/A'),
                        "Stops": getattr(cheapest_flight, 'stops', 0)
                    })

                    print(f"{getattr(cheapest_flight, 'price', 'N/A')}")
                    success = True
                    break
                else:
                    print(f"No flights found.")

            except Exception as error:
                error_msg = str(error)[:60].replace('\n', ' ')
                print(f"Error: [{error_msg}...]")

            wait_time = BASE_DELAY * (attempt + 1)
            print(f"   Cooling down {wait_time}s...")
            time.sleep(wait_time)

        if not success:
            print(f"Failed: {destination}")

        time.sleep(BASE_DELAY)

    df_flights = pd.DataFrame(extracted_records)
    df_flights.to_csv(output_file, index=False)
    print(f"\nSaved {len(df_flights)} records to {os.path.abspath(output_file)}")


Starting extraction for 2026-07-15 -> flight_data_peak.csv
--------------------------------------------------
[1/50] LAX -> CDG (Attempt 1/3)...
Error: [No flights found: Skip to main contentAccessibility feedback...]
   Cooling down 5s...
[1/50] LAX -> CDG (Attempt 2/3)...
Error: [No flights found: Skip to main contentAccessibility feedback...]
   Cooling down 10s...
[1/50] LAX -> CDG (Attempt 3/3)...
Error: [No flights found: Skip to main contentAccessibility feedback...]
   Cooling down 15s...
Failed: CDG


KeyboardInterrupt: 